# TQQQ Put Spread Backtest

Risk-managed CSP strategy with full wheel mechanics:
- **Sell weekly puts** at target delta (enter Monday open, expire Friday)
- **Buy protective puts** (4-8 weeks) for downside risk management
- **On assignment:** sell weekly covered calls until capital recovered or shares called away

Grid search: sell_delta × buy_delta × protective_duration × cc_delta → 1,215 combinations.
Data: Databento OPRA (institutional quality), greeks computed via py_vollib.


Code: Imports

In [1]:
import sys, os, warnings
from pathlib import Path
from datetime import date, timedelta
from itertools import product

import numpy as np
import pandas as pd
import yfinance as yf
import databento as db
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(override=True)

/Users/samuelminer/Projects/nissan_options/wheel_strategy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Code: Config

In [2]:
# Import project's GreeksCalculator
PROJECT_ROOT = Path("/Users/samuelminer/Projects/nissan_options/wheel_strategy")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "live_trading"))
from csp.data.greeks import GreeksCalculator

# ── Constants ───────────────────────────────────────────────────
SYMBOL = "TQQQ"
OPT_SYMBOL = "TQQQ.OPT"
START_DATE = "2026-02-01"
END_DATE = "2026-03-01"
RISK_FREE_RATE = 0.05

CACHE_DIR = PROJECT_ROOT / "tqqq" / "cache"
CACHE_DIR.mkdir(exist_ok=True)

DB_KEY = os.environ.get("TQQQ_DATABENTO_API_KEY")
client = db.Historical(DB_KEY)
greeks_calc = GreeksCalculator(risk_free_rate=RISK_FREE_RATE)

# ── Grid search ranges ──────────────────────────────────────────
SELL_DELTAS = np.round(np.arange(-0.10, -0.55, -0.05), 2)
BUY_DELTAS = np.round(np.arange(-0.10, -0.55, -0.05), 2)
BUY_DURATIONS = [4, 6, 8]  # weeks
CC_DELTAS = [-0.15, -0.20, -0.25, -0.30, -0.35]

print(f"Sell deltas:  {list(SELL_DELTAS)}")
print(f"Buy deltas:   {list(BUY_DELTAS)}")
print(f"Durations:    {BUY_DURATIONS}")
print(f"CC deltas:    {CC_DELTAS}")
print(f"Total combos: {len(SELL_DELTAS) * len(BUY_DELTAS) * len(BUY_DURATIONS) * len(CC_DELTAS)}")


Sell deltas:  [np.float64(-0.1), np.float64(-0.15), np.float64(-0.2), np.float64(-0.25), np.float64(-0.3), np.float64(-0.35), np.float64(-0.4), np.float64(-0.45), np.float64(-0.5)]
Buy deltas:   [np.float64(-0.1), np.float64(-0.15), np.float64(-0.2), np.float64(-0.25), np.float64(-0.3), np.float64(-0.35), np.float64(-0.4), np.float64(-0.45), np.float64(-0.5)]
Durations:    [4, 6, 8]
CC deltas:    [-0.15, -0.2, -0.25, -0.3, -0.35]
Total combos: 1215


Cell 3 — Code: TQQQ Underlying OHLC

In [3]:
cache_path = CACHE_DIR / "tqqq_daily.parquet"

if cache_path.exists():
    tqqq = pd.read_parquet(cache_path)
    print(f"Loaded cached TQQQ prices: {len(tqqq)} rows")
else:
    ticker = yf.Ticker(SYMBOL)
    hist = ticker.history(start=START_DATE, end=END_DATE, interval="1d")
    tqqq = hist[["Open", "Close"]].rename(columns={"Open": "open", "Close": "close"})
    tqqq.index = tqqq.index.tz_localize(None).normalize()
    tqqq.to_parquet(cache_path)
    print(f"Fetched and cached TQQQ prices: {len(tqqq)} rows")

print(f"Range: {tqqq.index.min().date()} → {tqqq.index.max().date()}")
tqqq.tail()


Fetched and cached TQQQ prices: 19 rows
Range: 2026-02-02 → 2026-02-27


,open,close
Date,,
2026-02-23,49.529999,48.240002
2026-02-24,48.450001,49.779999
2026-02-25,50.549999,51.869999
2026-02-26,51.630001,50.049999
2026-02-27,48.450001,49.520000


Cell 4 — Code: Cost Check — Definitions

In [4]:
# ── Cost Check + Fetch ───────────────────────────────────────────
from tqqq.fetch import check_cost, fetch_chunked

# Check costs first
check_cost(OPT_SYMBOL, "definition", START_DATE, END_DATE)
check_cost(OPT_SYMBOL, "ohlcv-1d", START_DATE, END_DATE)


Cost for definition (2026-02-01 → 2026-03-01): $0.04
Cost for ohlcv-1d (2026-02-01 → 2026-03-01): $3.32


3.323976695538

Cell 5 — Code: Fetch Definitions

In [5]:
# ── Fetch Definitions (puts + calls) ────────────────────────────
defs_raw = fetch_chunked(
    symbol=OPT_SYMBOL,
    schema="definition",
    start=START_DATE,
    end=END_DATE,
    cache_dir=CACHE_DIR,
    label="tqqq_definitions",
)

# strike_price is already in dollars after .to_df() — NO scaling needed
defs_raw["expiration_date"] = pd.to_datetime(defs_raw["expiration"], unit="ns").dt.date

# Dedup by instrument_id (definitions are re-published daily, same contract)
# Keep one row per instrument_id for joining with OHLCV
defs_clean = (
    defs_raw[["instrument_id", "raw_symbol", "strike_price", "expiration_date", "instrument_class"]]
    .drop_duplicates(subset=["instrument_id"])
    .rename(columns={"strike_price": "strike", "instrument_class": "option_type"})
)

print(f"\nDefinitions: {len(defs_raw):,} raw → {len(defs_clean):,} unique instrument_ids")
print(f"  Unique contracts (raw_symbol): {defs_clean['raw_symbol'].nunique():,}")
print(f"  Puts:  {(defs_clean['option_type'] == 'P').sum():,}")
print(f"  Calls: {(defs_clean['option_type'] == 'C').sum():,}")
print(f"  Strike range: ${defs_clean['strike'].min():.2f} – ${defs_clean['strike'].max():.2f}")
print(f"  Expiry range: {defs_clean['expiration_date'].min()} – {defs_clean['expiration_date'].max()}")


[tqqq_definitions] Fetching 1 chunks (workers=4)...


KeyboardInterrupt: 

Cell 6 — Fetch OHLCV:

In [ ]:
# ── Fetch OHLCV-1d (daily option prices, exchange-level) ────────
ohlcv_raw = fetch_chunked(
    symbol=OPT_SYMBOL,
    schema="ohlcv-1d",
    start=START_DATE,
    end=END_DATE,
    cache_dir=CACHE_DIR,
    label="tqqq_ohlcv_1d",
)

# ts_event is the index (trade date) — reset for merging
if ohlcv_raw.index.name == "ts_event":
    ohlcv_raw = ohlcv_raw.reset_index()

print(f"OHLCV raw: {len(ohlcv_raw):,} rows")
print(f"  Exchanges (publisher_ids): {ohlcv_raw['publisher_id'].nunique()}")
print(f"  Unique instrument_ids: {ohlcv_raw['instrument_id'].nunique():,}")
print(f"  Unique symbols: {ohlcv_raw['symbol'].nunique():,}")


[tqqq_ohlcv_1d] Loaded cached: 106,223 rows
OHLCV raw: 106,223 rows
  Exchanges (publisher_ids): 18
  Unique instrument_ids: 13,044
  Unique symbols: 1,515


Cell 7 — Aggregate exchanges + join:

In [ ]:
# ── Aggregate exchange-level OHLCV to consolidated daily bars ───
# Each exchange reports its own OHLC + volume for the same contract/day.
# Aggregate: sum volume, volume-weighted close, min low, max high.

ohlcv_raw["trade_date"] = pd.to_datetime(ohlcv_raw["ts_event"]).dt.normalize()

ohlcv_agg = (
    ohlcv_raw
    .assign(vol_x_close=lambda d: d["volume"] * d["close"])
    .groupby(["instrument_id", "trade_date"], as_index=False)
    .agg(
        volume=("volume", "sum"),
        vol_x_close=("vol_x_close", "sum"),
        high=("high", "max"),
        low=("low", "min"),
        open=("open", "first"),   # first exchange reported
    )
)
ohlcv_agg["close"] = (ohlcv_agg["vol_x_close"] / ohlcv_agg["volume"]).where(
    ohlcv_agg["volume"] > 0, ohlcv_agg["open"]
)
ohlcv_agg = ohlcv_agg.drop(columns=["vol_x_close"])

print(f"Aggregated: {len(ohlcv_raw):,} exchange rows → {len(ohlcv_agg):,} consolidated rows")
print(f"  Trade dates: {ohlcv_agg['trade_date'].dt.date.min()} – {ohlcv_agg['trade_date'].dt.date.max()}")


KeyError: 'ts_event'